# 07b · Calibración post hoc de probabilidades

Este cuaderno calibra **el ensemble final de cinco semillas** mediante un único escalado de temperatura. El parámetro se ajusta exclusivamente con la validación de desarrollo; las etiquetas externas no intervienen en el ajuste ni en la elección del método.

El resultado no sustituye al notebook 07: la evaluación externa sin calibrar permanece como análisis confirmatorio primario. Como la necesidad de calibrar se decidió después de inspeccionar el externo, las métricas calibradas se etiquetan como análisis secundario y post hoc.

La transformación es $p_k^{(cal)}=p_k^{1/T}/\sum_jp_j^{1/T}$. Al usar un único $T>0$, se conserva exactamente el `argmax`; por tanto, no pueden cambiar accuracy, balanced accuracy, macro-F1 ni la matriz de confusión.

Referencias: Guo et al. (2017), [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html); Van Calster et al. (2019), [Calibration: the Achilles heel of predictive analytics](https://bmcmedicine.biomedcentral.com/articles/10.1186/s12916-019-1466-7); Cawley y Talbot (2010), [On Over-fitting in Model Selection](https://jmlr.org/papers/v11/cawley10a.html).

## Contrato del análisis

- Familia fijada: **temperature scaling global del ensemble**; no se comparan familias.
- Objetivo de ajuste: log-loss multiclase en desarrollo.
- Estimación interna: validación cruzada repetida, estratificada por cohorte × clase.
- Ajuste final: toda la validación de desarrollo.
- Aplicación externa: una sola transformación de las probabilidades ya guardadas por el notebook 07.
- Salida independiente: `results_v8/07b_calibration_posthoc/`; nunca se modifica `07_external_locked`.

La validación de desarrollo ya intervino en *early stopping*. La validación cruzada de este cuaderno controla el sobreajuste del calibrador de un parámetro, pero no convierte el conjunto de desarrollo en una evaluación completamente independiente del modelo. Esta es otra razón para mantener la calibración como análisis secundario.

In [ ]:
from pathlib import Path
import gc, hashlib, json, sys

import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import log_loss
from sklearn.model_selection import RepeatedStratifiedKFold

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'intergate').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Ejecute este notebook dentro de InterGATE_V8_1.')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.ablation import build_pruned_model_from_bundle, load_bundle
from intergate.calibration import TemperatureCalibrator, log_probabilities
from intergate.config import CFG
from intergate.data import ExpressionDataset
from intergate.losses import compute_metrics_full
from intergate.runtime_setup import (
    assert_transport_protocol, assert_v8_protocol, prepare_runtime,
)
from intergate.training import predict_proba_xh_mode

assert_v8_protocol(CFG)
assert_transport_protocol(CFG)

ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / 'INTERGATE_V8'
RESULTS = PROJECT_ROOT / 'results_v8'
RAW_EXTERNAL_DIR = RESULTS / '07_external_locked'
OUT = RESULTS / '07b_calibration_posthoc'
OUT.mkdir(parents=True, exist_ok=True)

LOCK = ARTIFACTS / 'protocol_lock_v8.json'
RAW_EXTERNAL = RAW_EXTERNAL_DIR / 'external_predictions.csv.gz'
RAW_COMPLETE = RAW_EXTERNAL_DIR / 'external_evaluation_COMPLETE.json'
CAL_COMPLETE = OUT / 'calibration_COMPLETE.json'

print('Proyecto:', PROJECT_ROOT)
print('Artefactos:', RUN_ROOT)
print('Salida nueva:', OUT)

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

required = [LOCK, RAW_EXTERNAL, RAW_COMPLETE]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Faltan entradas del flujo 06/07:\n- ' + '\n- '.join(missing))

lock = json.loads(LOCK.read_text(encoding='utf-8'))
raw_complete = json.loads(RAW_COMPLETE.read_text(encoding='utf-8'))
if raw_complete.get('status') != 'complete':
    raise RuntimeError('El notebook 07 no consta como completo.')
if raw_complete.get('protocol_lock_sha256') != sha256(LOCK):
    raise RuntimeError('El resultado 07 no corresponde al protocol lock actual.')
if CAL_COMPLETE.is_file():
    display(pd.read_csv(OUT / 'external_calibration_metrics.csv'))
    raise SystemExit('La calibración ya estaba completa; no se ha sobrescrito.')

changed = []
for path_text, expected in lock['files'].items():
    path = Path(path_text)
    if not path.is_file() or sha256(path) != expected['sha256']:
        changed.append(path_text)
if changed:
    raise RuntimeError('Cambió un archivo congelado después del notebook 06:\n- ' + '\n- '.join(changed))

RAW_EXTERNAL_SHA256 = sha256(RAW_EXTERNAL)
print('Integridad 06/07 correcta.')
print('SHA-256 predicciones externas crudas:', RAW_EXTERNAL_SHA256)
print('El notebook 07 y sus resultados permanecerán intactos.')

## Reconstrucción de predicciones de validación

Se realiza únicamente inferencia con los cinco checkpoints ya entrenados. `include_external=False` impide cargar expresión o etiquetas externas durante el ajuste de la temperatura. No se actualiza ningún peso del modelo.

In [ ]:
import torch

runtime = prepare_runtime(CFG, include_external=False)
if runtime.get('external_opened') is not False or len(runtime['test_idx']) != 0:
    raise RuntimeError('El runtime de calibración abrió el bloque externo.')

X_FULL = np.asarray(runtime['X_scaled'], dtype=np.float32)
X_H = np.asarray(runtime['X_h'], dtype=np.float32)
Y = np.asarray(runtime['y'], dtype=np.int64)
COHORT = np.asarray(runtime['cohort']).astype(str)
VAL_IDX = np.asarray(runtime['val_idx'], dtype=np.int64)
SEEDS = tuple(map(int, CFG.STABILITY_SEEDS))
CLASSES = [str(name) for name in runtime['classes']]
LABEL_MAP = runtime['label_map']
P_COLS = [f'p_{name}' for name in CLASSES]

def predict_validation_seed(seed):
    bundle = load_bundle(str(RUN_ROOT / f'seed_{int(seed)}'))
    model = build_pruned_model_from_bundle(bundle, device=CFG.DEVICE)
    model.eval()
    nodes = np.asarray(bundle['graph']['nodes_used_full'], dtype=np.int64)
    flags = bundle.get('model', {}).get('model_flags', {})
    if bool(flags.get('tab_full_gene_universe', False)):
        X_comp = np.hstack([X_FULL[:, nodes], X_FULL]).astype(np.float32)
    else:
        X_comp = X_FULL[:, nodes]
    loader = torch.utils.data.DataLoader(
        ExpressionDataset(X_comp, X_H, Y, VAL_IDX),
        batch_size=int(CFG.BATCH_SIZE), shuffle=False, num_workers=0,
    )
    with torch.no_grad():
        proba, truth = predict_proba_xh_mode(
            model, None, loader, CFG.DEVICE, xh_mode='orig',
        )
    if not np.array_equal(np.asarray(truth, dtype=int), Y[VAL_IDX]):
        raise RuntimeError(f'Semilla {seed}: se alteró el orden de validación.')
    frame = pd.DataFrame({
        'seed': int(seed), 'sample_index': VAL_IDX,
        'cohort': COHORT[VAL_IDX], 'y_true': Y[VAL_IDX],
    })
    for class_id, column in enumerate(P_COLS):
        frame[column] = np.asarray(proba)[:, class_id]
    frame['y_pred'] = np.asarray(proba).argmax(axis=1)
    del model, loader, X_comp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return frame

validation_parts = []
for seed in SEEDS:
    print(f'Inferencia de validación, semilla {seed}...', end=' ', flush=True)
    validation_parts.append(predict_validation_seed(seed))
    print('ok')
validation_predictions = pd.concat(validation_parts, ignore_index=True)
validation_predictions.to_csv(
    OUT / 'development_validation_predictions_by_seed.csv.gz',
    index=False, compression='gzip',
)
print('Predicciones de desarrollo:', validation_predictions.shape)

In [ ]:
def validate_prediction_frame(frame, sample_column, expected_seeds):
    if frame.duplicated([sample_column, 'seed']).any():
        raise RuntimeError(f'Hay duplicados {sample_column} × seed.')
    observed = tuple(sorted(map(int, frame['seed'].unique())))
    if observed != tuple(sorted(expected_seeds)):
        raise RuntimeError(f'Semillas inesperadas: {observed}; esperadas={expected_seeds}.')
    counts = frame.groupby(sample_column)['seed'].nunique()
    if not (counts == len(expected_seeds)).all():
        raise RuntimeError('No todos los individuos tienen las cinco predicciones.')
    p = frame[P_COLS].to_numpy(dtype=float)
    if np.any(~np.isfinite(p)) or np.any(p < 0):
        raise RuntimeError('Probabilidades no válidas.')
    if not np.allclose(p.sum(axis=1), 1.0, atol=1e-5):
        raise RuntimeError('Las probabilidades no suman uno.')

validate_prediction_frame(validation_predictions, 'sample_index', SEEDS)
val_ensemble = (
    validation_predictions
    .groupby(['sample_index', 'cohort', 'y_true'], as_index=False)[P_COLS]
    .mean()
    .sort_values('sample_index')
    .reset_index(drop=True)
)
val_ensemble['y_pred'] = val_ensemble[P_COLS].to_numpy().argmax(axis=1)

external_predictions = pd.read_csv(RAW_EXTERNAL)
validate_prediction_frame(external_predictions, 'test_index', SEEDS)
external_ensemble = (
    external_predictions
    .groupby(['test_index', 'cohort', 'y_true'], as_index=False)[P_COLS]
    .mean()
    .sort_values('test_index')
    .reset_index(drop=True)
)
external_ensemble['y_pred'] = external_ensemble[P_COLS].to_numpy().argmax(axis=1)
if len(external_ensemble) != int(raw_complete['n_external_samples']):
    raise RuntimeError('El número de individuos externos no coincide con el manifiesto 07.')

print(f'Ensemble desarrollo: n={len(val_ensemble)}')
print(f'Ensemble externo guardado: n={len(external_ensemble)}')
display(val_ensemble.groupby(['cohort', 'y_true']).size().unstack(fill_value=0))

## Validación cruzada interna del calibrador

La validación cruzada estima si el único parámetro generaliza dentro de desarrollo. No selecciona entre métodos: identidad y temperatura se muestran únicamente como comparación preespecificada. El calibrador definitivo se ajustará después con toda la validación.

In [ ]:
def multiclass_brier(y_true, proba):
    y_true = np.asarray(y_true, dtype=int)
    one_hot = np.eye(proba.shape[1], dtype=float)[y_true]
    return float(np.mean(np.sum((np.asarray(proba) - one_hot) ** 2, axis=1)))

def calibration_table(y_true, proba, n_bins=15, class_id=None):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    if class_id is None:
        confidence = proba.max(axis=1)
        event = (proba.argmax(axis=1) == y_true).astype(float)
    else:
        confidence = proba[:, int(class_id)]
        event = (y_true == int(class_id)).astype(float)
    edges = np.linspace(0.0, 1.0, int(n_bins) + 1)
    bin_id = np.clip(np.digitize(confidence, edges[1:-1], right=True), 0, n_bins - 1)
    rows = []
    for b in range(int(n_bins)):
        mask = bin_id == b
        if not mask.any():
            continue
        rows.append({
            'bin': b, 'lower': edges[b], 'upper': edges[b + 1],
            'n': int(mask.sum()),
            'mean_probability': float(confidence[mask].mean()),
            'observed_frequency': float(event[mask].mean()),
        })
    return pd.DataFrame(rows)

def ece(y_true, proba, n_bins=15):
    table = calibration_table(y_true, proba, n_bins=n_bins)
    return float(
        np.sum(table['n'] * np.abs(
            table['observed_frequency'] - table['mean_probability']
        )) / table['n'].sum()
    )

def probability_metrics(y_true, proba):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    hard = compute_metrics_full(y_true, proba, LABEL_MAP)
    return {
        **hard,
        'log_loss': float(log_loss(y_true, proba, labels=np.arange(len(CLASSES)))),
        'brier_multiclass': multiclass_brier(y_true, proba),
        'ece15': ece(y_true, proba, n_bins=15),
        'mean_confidence': float(proba.max(axis=1).mean()),
    }

P_VAL = val_ensemble[P_COLS].to_numpy(dtype=float)
Y_VAL = val_ensemble['y_true'].to_numpy(dtype=int)
joint_stratum = (
    val_ensemble['cohort'].astype(str) + '::' + val_ensemble['y_true'].astype(str)
).to_numpy()
min_stratum = int(pd.Series(joint_stratum).value_counts().min())
N_SPLITS = min(5, min_stratum)
if N_SPLITS < 2:
    raise RuntimeError('No hay al menos dos observaciones en cada estrato cohorte × clase.')

splitter = RepeatedStratifiedKFold(
    n_splits=N_SPLITS, n_repeats=10, random_state=20260827,
)
cv_rows = []
for fold, (fit_rows, held_rows) in enumerate(splitter.split(P_VAL, joint_stratum)):
    calibrator = TemperatureCalibrator(len(CLASSES))
    calibrator.fit(log_probabilities(P_VAL[fit_rows]), Y_VAL[fit_rows])
    calibrated = softmax(
        calibrator.transform(log_probabilities(P_VAL[held_rows])), axis=1,
    )
    for state, probabilities in [('uncalibrated', P_VAL[held_rows]), ('temperature', calibrated)]:
        cv_rows.append({
            'fold': int(fold), 'state': state,
            'temperature_fit': float(calibrator.parameters()['temperature']),
            'n_fit': int(len(fit_rows)), 'n_held_out': int(len(held_rows)),
            **probability_metrics(Y_VAL[held_rows], probabilities),
        })

cv_detail = pd.DataFrame(cv_rows)
cv_summary = (
    cv_detail.groupby('state', sort=False)
    .agg(
        log_loss_mean=('log_loss', 'mean'), log_loss_sd=('log_loss', 'std'),
        brier_mean=('brier_multiclass', 'mean'), brier_sd=('brier_multiclass', 'std'),
        ece15_mean=('ece15', 'mean'), ece15_sd=('ece15', 'std'),
        macro_f1_mean=('macro_f1', 'mean'), macro_f1_sd=('macro_f1', 'std'),
        n_folds=('fold', 'count'),
    ).reset_index()
)
cv_detail.to_csv(OUT / 'development_temperature_cv_detail.csv', index=False)
cv_summary.to_csv(OUT / 'development_temperature_cv_summary.csv', index=False)
print(f'CV: {N_SPLITS} folds × 10 repeticiones; estratificación cohorte × clase.')
display(cv_summary.round(5))

## Ajuste final en desarrollo y aplicación al externo guardado

A partir de esta celda, (T) queda congelada. El fichero externo solo se utiliza para cuantificar, de forma secundaria, el efecto de esa transformación ya fijada.

In [ ]:
final_calibrator = TemperatureCalibrator(len(CLASSES))
final_calibrator.fit(log_probabilities(P_VAL), Y_VAL)
TEMPERATURE = float(final_calibrator.parameters()['temperature'])

P_VAL_CAL = softmax(
    final_calibrator.transform(log_probabilities(P_VAL)), axis=1,
)
P_EXT = external_ensemble[P_COLS].to_numpy(dtype=float)
Y_EXT = external_ensemble['y_true'].to_numpy(dtype=int)
P_EXT_CAL = softmax(
    final_calibrator.transform(log_probabilities(P_EXT)), axis=1,
)

if not np.array_equal(P_VAL.argmax(axis=1), P_VAL_CAL.argmax(axis=1)):
    raise RuntimeError('La temperatura cambió alguna clase predicha en desarrollo.')
if not np.array_equal(P_EXT.argmax(axis=1), P_EXT_CAL.argmax(axis=1)):
    raise RuntimeError('La temperatura cambió alguna clase predicha externa.')

val_uncal = val_ensemble.copy()
val_cal = val_ensemble.copy()
external_uncal = external_ensemble.copy()
external_cal = external_ensemble.copy()
for class_id, column in enumerate(P_COLS):
    val_cal[column] = P_VAL_CAL[:, class_id]
    external_cal[column] = P_EXT_CAL[:, class_id]
val_cal['y_pred'] = P_VAL_CAL.argmax(axis=1)
external_cal['y_pred'] = P_EXT_CAL.argmax(axis=1)

val_uncal.to_csv(OUT / 'development_ensemble_uncalibrated.csv.gz', index=False, compression='gzip')
val_cal.to_csv(OUT / 'development_ensemble_calibrated.csv.gz', index=False, compression='gzip')
external_uncal.to_csv(OUT / 'external_ensemble_uncalibrated.csv.gz', index=False, compression='gzip')
external_cal.to_csv(OUT / 'external_ensemble_calibrated.csv.gz', index=False, compression='gzip')

print(f'Temperatura global congelada: T={TEMPERATURE:.6f}')
print('Clases predichas modificadas: 0')

In [ ]:
metric_rows = []
for dataset, frame, raw_p, calibrated_p in [
    ('development_validation', val_ensemble, P_VAL, P_VAL_CAL),
    ('external_secondary_posthoc', external_ensemble, P_EXT, P_EXT_CAL),
]:
    scopes = [('all', np.ones(len(frame), dtype=bool))]
    scopes += [
        (f'cohort::{cohort}', frame['cohort'].to_numpy() == cohort)
        for cohort in sorted(frame['cohort'].unique())
    ]
    for scope, mask in scopes:
        for state, probabilities in [('uncalibrated', raw_p), ('temperature', calibrated_p)]:
            metric_rows.append({
                'dataset': dataset, 'scope': scope, 'state': state,
                'n': int(mask.sum()),
                **probability_metrics(frame.loc[mask, 'y_true'], probabilities[mask]),
            })
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUT / 'external_calibration_metrics.csv', index=False)

reliability_rows = []
classwise_rows = []
for state, probabilities in [('uncalibrated', P_EXT), ('temperature', P_EXT_CAL)]:
    top = calibration_table(Y_EXT, probabilities, n_bins=15)
    top.insert(0, 'state', state)
    reliability_rows.append(top)
    for class_id, class_name in enumerate(CLASSES):
        table = calibration_table(Y_EXT, probabilities, n_bins=15, class_id=class_id)
        table.insert(0, 'class_name', class_name)
        table.insert(0, 'state', state)
        classwise_rows.append(table)
reliability = pd.concat(reliability_rows, ignore_index=True)
classwise_reliability = pd.concat(classwise_rows, ignore_index=True)
reliability.to_csv(OUT / 'external_reliability_toplabel.csv', index=False)
classwise_reliability.to_csv(OUT / 'external_reliability_classwise.csv', index=False)

overall = metrics.loc[metrics['scope'].eq('all')].copy()
display(overall[[
    'dataset', 'state', 'n', 'acc', 'bal_acc', 'macro_f1', 'auc_macro_ovr',
    'log_loss', 'brier_multiclass', 'ece15', 'mean_confidence',
]].round(5))

## Incertidumbre externa condicional a la temperatura fijada

Se usan 2.000 réplicas bootstrap estratificadas por cohorte × clase. Los intervalos cuantifican la variación muestral externa condicionada al (T) ajustado en desarrollo; no convierten el análisis post hoc en confirmatorio. Un delta negativo favorece la calibración para log-loss, Brier y ECE.

In [ ]:
BOOTSTRAP_REPLICATES = 2000
rng = np.random.default_rng(20260827)
strata = (
    external_ensemble['cohort'].astype(str) + '::' + external_ensemble['y_true'].astype(str)
).to_numpy()
stratum_indices = [np.flatnonzero(strata == value) for value in np.unique(strata)]

def calibration_loss_triplet(y_true, proba):
    return {
        'log_loss': float(log_loss(y_true, proba, labels=np.arange(len(CLASSES)))),
        'brier_multiclass': multiclass_brier(y_true, proba),
        'ece15': ece(y_true, proba, n_bins=15),
    }

bootstrap_rows = []
for replicate in range(BOOTSTRAP_REPLICATES):
    sampled = np.concatenate([
        rng.choice(index, size=len(index), replace=True) for index in stratum_indices
    ])
    raw_metrics = calibration_loss_triplet(Y_EXT[sampled], P_EXT[sampled])
    cal_metrics = calibration_loss_triplet(Y_EXT[sampled], P_EXT_CAL[sampled])
    bootstrap_rows.append({
        'replicate': replicate,
        **{f'delta_{name}': cal_metrics[name] - raw_metrics[name] for name in raw_metrics},
    })
bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(OUT / 'external_calibration_bootstrap.csv.gz', index=False, compression='gzip')

observed_raw = calibration_loss_triplet(Y_EXT, P_EXT)
observed_cal = calibration_loss_triplet(Y_EXT, P_EXT_CAL)
bootstrap_summary = []
for name in observed_raw:
    values = bootstrap[f'delta_{name}'].to_numpy()
    bootstrap_summary.append({
        'metric': name, 'uncalibrated': observed_raw[name],
        'temperature': observed_cal[name],
        'delta_temperature_minus_uncalibrated': observed_cal[name] - observed_raw[name],
        'delta_ci95_low': float(np.percentile(values, 2.5)),
        'delta_ci95_high': float(np.percentile(values, 97.5)),
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    })
bootstrap_summary = pd.DataFrame(bootstrap_summary)
bootstrap_summary.to_csv(OUT / 'external_calibration_bootstrap_summary.csv', index=False)
display(bootstrap_summary.round(6))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.4, 5.0))
ax.plot([0, 1], [0, 1], '--', color='0.55', linewidth=1.2, label='Ideal')
for state, color, label in [
    ('uncalibrated', '#B33A3A', 'Sin calibrar'),
    ('temperature', '#2468A2', 'Temperatura'),
]:
    table = reliability.loc[reliability['state'].eq(state)]
    ax.plot(table['mean_probability'], table['observed_frequency'], 'o-',
            color=color, linewidth=1.8, markersize=5, label=label)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Confianza media',
       ylabel='Proporción correcta',
       title='Calibración externa del ensemble (análisis post hoc)')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / 'external_reliability_toplabel.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT / 'external_reliability_toplabel.pdf', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(9.0, 8.0), sharex=True, sharey=True)
for class_name, ax in zip(CLASSES, axes.ravel()):
    ax.plot([0, 1], [0, 1], '--', color='0.65', linewidth=1.0)
    for state, color, label in [
        ('uncalibrated', '#B33A3A', 'Sin calibrar'),
        ('temperature', '#2468A2', 'Temperatura'),
    ]:
        table = classwise_reliability.loc[
            classwise_reliability['state'].eq(state)
            & classwise_reliability['class_name'].eq(class_name)
        ]
        ax.plot(table['mean_probability'], table['observed_frequency'], 'o-',
                color=color, linewidth=1.5, markersize=4, label=label)
    ax.set_title(class_name)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
axes[1, 0].set_xlabel('Probabilidad predicha')
axes[1, 1].set_xlabel('Probabilidad predicha')
axes[0, 0].set_ylabel('Frecuencia observada')
axes[1, 0].set_ylabel('Frecuencia observada')
axes[0, 0].legend(frameon=False)
fig.suptitle('Calibración externa por clase (análisis post hoc)', y=1.01)
fig.tight_layout()
fig.savefig(OUT / 'external_reliability_classwise.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT / 'external_reliability_classwise.pdf', bbox_inches='tight')
plt.show()

In [ ]:
manifest = {
    'status': 'complete',
    'analysis': 'posthoc_probability_calibration',
    'protocol_version': str(CFG.TRANSPORT_PROTOCOL_VERSION),
    'uncalibrated_external_role': 'primary_confirmatory',
    'calibrated_external_role': 'secondary_posthoc',
    'family': 'temperature',
    'scope': 'single_global_temperature_on_five_seed_probability_ensemble',
    'objective': 'multiclass_log_loss',
    'n_calibration_parameters': 1,
    'temperature': TEMPERATURE,
    'fit_split': 'development_validation_only',
    'development_validation_also_used_for_model_early_stopping': True,
    'fit_n': int(len(val_ensemble)),
    'external_n': int(len(external_ensemble)),
    'classes': CLASSES,
    'seeds': list(SEEDS),
    'cross_validation': {
        'splitter': 'RepeatedStratifiedKFold',
        'stratum': 'development_cohort_x_class',
        'n_splits': int(N_SPLITS), 'n_repeats': 10,
        'random_state': 20260827,
    },
    'external_used_to_fit_temperature': False,
    'external_used_to_compare_calibration_families': False,
    'external_result_had_already_been_inspected': True,
    'external_triggered_posthoc_calibration_analysis': True,
    'argmax_changed_in_development': False,
    'argmax_changed_in_external': False,
    'raw_protocol_lock_sha256': sha256(LOCK),
    'raw_external_predictions_sha256': RAW_EXTERNAL_SHA256,
    'validation_predictions_sha256': sha256(
        OUT / 'development_validation_predictions_by_seed.csv.gz'
    ),
    'bootstrap': {
        'replicates': BOOTSTRAP_REPLICATES,
        'stratum': 'external_cohort_x_class',
        'random_state': 20260827,
        'interpretation': 'external_sampling_uncertainty_conditional_on_fitted_temperature',
    },
    'references': [
        'Guo et al. ICML 2017; PMLR 70:1321-1330',
        'Van Calster et al. BMC Medicine 2019;17:230',
        'Cawley and Talbot. JMLR 2010;11:2079-2107',
    ],
}
CAL_COMPLETE.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8',
)
print(json.dumps({
    'status': manifest['status'], 'temperature': TEMPERATURE,
    'fit_split': manifest['fit_split'],
    'external_role': manifest['calibrated_external_role'],
}, indent=2, ensure_ascii=False))
print('Calibración completa:', OUT)

## Interpretación

1. La evidencia principal sobre discriminación y clasificación sigue siendo la del notebook 07 sin calibrar.
2. Para valorar la calibración, priorice log-loss y Brier; ECE-15 depende del binning y se informa como diagnóstico complementario.
3. Si la validación cruzada interna o el externo no mejoran, se informa el resultado sin probar otra familia.
4. La temperatura no corrige confusiones LumA–LumB ni sobrepredicción de HER2; esas modificaciones requerirían umbrales o sesgos por clase ajustados en un protocolo nuevo.
5. Reentrenar con el mismo externo no restablecería su independencia. Una confirmación plenamente prospectiva de la versión calibrada necesita una cohorte externa nueva y no inspeccionada.